**The Scenario:** The company database crashed, and the backup data is a mess. You must clean the "Orders" dataset.
- Handling Missing Data: The dataset is riddled with NaNs. You must use .dropna() to remove rows completely missing an Order ID, .fillna() to replace missing shipping states and transaction with "Unknown" that were lost during export.
- Handling Duplicates: A glitch caused users to be charged twice. You must use .duplicated() to find identical transactions and .drop_duplicates(keep='first') to refund/remove the errors.

In [71]:
import pandas as pd

df = pd.read_csv("ecommerce.csv")
df.head()
df.info()
# print(df.isna())

df_drop_subset = df.dropna(subset=["id"])
# print(df_drop_subset)

df["Shipping State"] = df["shipping state"].fillna("Unknown")
df["trx"] = df["trx"].fillna("Unknown")

duplicates = df[df.duplicated()]
print(duplicates)
df = df.drop_duplicates(keep='first')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              930 non-null    float64
 1   customer name   1100 non-null   object 
 2   email           1100 non-null   object 
 3   purchase date   1100 non-null   object 
 4   price           1100 non-null   object 
 5   trx             1028 non-null   object 
 6   shipping state  856 non-null    object 
dtypes: float64(1), object(6)
memory usage: 60.3+ KB
         id        customer name                        email purchase date  \
1     689.0         kimmy haborn          khabornj4@google.es       pending   
2     414.0    matthias calcutt       mcalcuttbh@freewebs.com    11/17/2025   
3     789.0    ALGERNON NIAVES     aniaveslw@wunderground.com    2025-09-20   
5     760.0       hayes kleewein  hkleeweinl3@kickstarter.com    2025-12-30   
6      97.0    SUSANNE CUMMINS          s

**Data Type Conversions:** 
- "Price" is imported as a string (e.g., "$1,200.50"). You must remove the symbols and use pd.to_numeric() to turn it into a float.
- Replace the “pending” and “unknown” with the previous datetime
- Convert “purchase date" from messy text into a proper datetime object using pd.to_datetime().
- Convert the “purchase date" format from various format into this format “M/D/Y”

In [60]:
df["price"] = (df["price"].replace(r"[\$,]", "", regex=True))
df["price"] = pd.to_numeric(df["price"], errors="coerce")
print(df["price"])

df["purchase date"] = df["purchase date"].replace(["pending", "unknown"], pd.NA)
df["purchase date"] = df["purchase date"].fillna(method="ffill")
df["purchase date"] = pd.to_datetime(df["purchase date"], errors="coerce")
df["purchase date"] = df["purchase date"].dt.strftime("%-m/%-d/%Y")
print(df["purchase date"])

0       460.25
1       539.17
2       534.17
3       345.84
4       283.29
         ...  
1093    380.12
1094    699.39
1095    127.81
1096    500.37
1099    173.04
Name: price, Length: 1000, dtype: float64
0       2025-07-30 00:00:00
1       2025-07-30 00:00:00
2       2025-07-30 00:00:00
3       2025-07-30 00:00:00
4       2025-07-30 00:00:00
               ...         
1093    2025-05-21 00:00:00
1094    2025-05-21 00:00:00
1095    2025-05-21 00:00:00
1096    2025-03-17 00:00:00
1099    2026-01-10 00:00:00
Name: purchase date, Length: 1000, dtype: object


C:\Users\Dell\AppData\Local\Temp\ipykernel_1860\3612364301.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price"] = (df["price"].replace(r"[\$,]", "", regex=True))
C:\Users\Dell\AppData\Local\Temp\ipykernel_1860\3612364301.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price"] = pd.to_numeric(df["price"], errors="coerce")
C:\Users\Dell\AppData\Local\Temp\ipykernel_1860\3612364301.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

**String Manipulation:** The "Names" column has chaotic capitalization and extra spaces. You use .str.title() and .str.strip() to fix this. You must also extract the email domain (e.g., extracting "gmail.com" from "user@gmail.com") using .str.split().

In [69]:
df["customer name"] = df["customer name"].str.strip().str.title()

df["email"] = df["email"].str.split("@").str[-1]

print("\nCleaned Data Preview:\n", df.head())
print("\nData Info:\n")
df.info()

df.to_csv("cleaned_ecommerce.csv", index=False)


Cleaned Data Preview:
       id      customer name             email        purchase date   price  \
0    NaN      Port Beckwith     geocities.com  2025-07-30 00:00:00  460.25   
1  689.0       Kimmy Haborn         google.es  2025-07-30 00:00:00  539.17   
2  414.0   Matthias Calcutt      freewebs.com  2025-07-30 00:00:00  534.17   
3  789.0    Algernon Niaves  wunderground.com  2025-07-30 00:00:00  345.84   
4    NaN  Kynthia Mcquirter    yellowbook.com  2025-07-30 00:00:00  283.29   

                                    trx shipping state Shipping State  
0  548f8041-6317-4388-8dce-fe6de027505c      Argentina      Argentina  
1                               Unknown         Panama         Panama  
2  4a22264d-4c87-4f9d-b4a5-99e343cb768e         Brazil         Brazil  
3  83a9aa51-0a17-4fd8-a7c3-46a87c019f7b    Philippines    Philippines  
4  29c1f562-82cb-40d1-97aa-a467feeb24d8      Indonesia      Indonesia  

Data Info:

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 t

C:\Users\Dell\AppData\Local\Temp\ipykernel_1860\2938221865.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["customer name"] = df["customer name"].str.strip().str.title()
C:\Users\Dell\AppData\Local\Temp\ipykernel_1860\2938221865.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["email"] = df["email"].str.split("@").str[-1]
